# Cost of living

Trying to answer: 
<p>How much does a family need to make a year for this - Starting a family in a nice town, with enough financial success to be generally free from worry. The traditional idea of "making it". </p>

## Is the American Dream still achievable?

**Question:** Has income growth kept up with the cost of living — broadly, and in the big-ticket categories (housing, healthcare, higher ed, food)?

**Approach:** Index per-capita personal income and PCE price indexes to **2000 = 100**. A series above 100 has grown since 2000; if income sits above a cost-of-living series, that cost has gotten *more affordable* relative to the average paycheck. If it sits below, the squeeze is real.

Data:
- Income → `cleaned_income_with_state.csv` (BEA per-capita personal income, by state)
- Prices → `Price_Indexes_for_Personal_Consumption_Expenditures_by_Function.csv` (BEA T2.5.4, base 2017=100)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

income = pd.read_csv("../../data/cleaned/cleaned_income_with_state.csv")
prices = pd.read_csv("../../data/cleaned/Price_Indexes_for_Personal_Consumption_Expenditures_by_Function.csv")

# strip BEA footnote markers like \1\, \2\ from category labels
prices["category"] = prices["category"].str.replace(r"\\\d+\\", "", regex=True).str.strip()

year_cols = [c for c in prices.columns if str(c).isdigit()]
print("price years:", year_cols[0], "→", year_cols[-1])
print("income cols tail:", list(income.columns[-5:]))

In [ ]:
# Build US per-capita income by aggregating states:
#   total personal income ($M)  /  total population  * 1,000,000
yr_income = [c for c in income.columns if str(c).isdigit()]

pi  = income[income["LineCode"] == 1].set_index("GeoName")[yr_income].sum()   # $ millions
pop = income[income["LineCode"] == 2].set_index("GeoName")[yr_income].sum()   # persons

us_pci = (pi * 1_000_000) / pop          # dollars per person
us_pci.index = us_pci.index.astype(int)
us_pci = us_pci.sort_index()
us_pci.tail()

In [ ]:
# Pick the cost-of-living categories most tied to a family's budget
picks = {
    "Personal consumption expenditures":                                   "Overall cost of living (PCE)",
    "Housing":                                                             "Housing",
    "Rental of tenant-occupied nonfarm housing":                           "Rent (tenant)",
    "Health":                                                              "Healthcare",
    "Higher education":                                                    "Higher education",
    "Food and nonalcoholic beverages purchased for off-premises consumption": "Groceries",
}

cost = prices[prices["category"].isin(picks)].set_index("category")[year_cols].T
cost.index = cost.index.astype(int)
cost = cost.rename(columns=picks).sort_index()

# Align both series on a common year range
base = 2000
start, end = base, min(us_pci.index.max(), cost.index.max())
cost = cost.loc[start:end]
inc  = us_pci.loc[start:end]

# Re-index everything so 2000 = 100 (growth comparison)
cost_idx = cost / cost.loc[base] * 100
inc_idx  = inc  / inc.loc[base]  * 100

cost_idx.tail(), inc_idx.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))

# income bold; costs thinner
ax.plot(inc_idx.index, inc_idx.values, color="black", lw=3,
        label="Per-capita income (US)", zorder=5)

colors = {
    "Overall cost of living (PCE)": "#888888",
    "Housing":                      "#1f77b4",
    "Rent (tenant)":                "#17becf",
    "Healthcare":                   "#d62728",
    "Higher education":             "#9467bd",
    "Groceries":                    "#2ca02c",
}
for col in cost_idx.columns:
    ax.plot(cost_idx.index, cost_idx[col], lw=1.8, label=col, color=colors.get(col))

ax.axhline(100, color="gray", ls="--", lw=0.8)
ax.set_title(f"Income vs Cost of Living in the US  (indexed, {base} = 100)")
ax.set_xlabel("Year"); ax.set_ylabel(f"Index ({base} = 100)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# "Real" buying power by category: how much a year of income buys of each thing,
# indexed so 2000 = 100.  Above 100 = more affordable than in 2000.  Below 100 = harder.
affordability = inc_idx.to_frame("income").join(cost_idx)
for col in cost_idx.columns:
    affordability[col] = affordability["income"] / affordability[col] * 100
affordability = affordability.drop(columns="income")

fig, ax = plt.subplots(figsize=(12, 6))
for col in affordability.columns:
    ax.plot(affordability.index, affordability[col], lw=1.8, label=col, color=colors.get(col))
ax.axhline(100, color="black", ls="--", lw=1, label=f"{base} baseline")
ax.set_title(f"How much income buys, relative to {base}")
ax.set_xlabel("Year"); ax.set_ylabel(f"Affordability index ({base} = 100)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\n=== Latest affordability (vs 2000) ===")
print(affordability.iloc[-1].round(1).sort_values())

### Reading the charts (2000 → 2024)

Per-capita income grew to **~240** (2000 = 100). Where did that buying power go?

| Category | Price index 2024 (2000=100) | Income ran ahead? |
|---|---:|---|
| Groceries | 183 | ✅ yes, by a lot |
| Healthcare | 175 | ✅ yes |
| Overall PCE (headline inflation) | 168 | ✅ yes |
| Housing | 212 | ✅ barely |
| Rent (tenant) | 228 | ⚠️ barely |
| **Higher education** | **282** | ❌ **no — college outpaced paychecks** |

**Takeaway:** On paper, the American Dream math still works *on average* — nominal income grew faster than overall cost of living, so real buying power is higher than in 2000. But the two things that define "making it" for a family — **owning / renting a decent home** and **paying for college** — have eaten most of that gain. Higher education is the only category that has outright *gotten harder* to afford. That lines up with how the dream *feels* unattainable even while the aggregate numbers say it isn't.

Caveat: per-capita income is a mean, not a median — it's pulled up by top earners, so a typical household's experience is worse than what this chart implies.